In [1]:
import pandas as pd
from pathlib import Path

from databricks_utils import get_delta_table_mm

In [2]:
# --- notebook-wide: don't wrap dataframe cells (applies to plain df reprs too,
# not just tables run through style_df) ---
from IPython.display import HTML, display

display(HTML("<style>.dataframe td, .dataframe th { white-space: nowrap; }</style>"))

In [3]:
# --- reusable display helper: comma-separated, centered, () for negatives, no wrap.
# Default 1 decimal place; pass decimal_overrides={"col": n, ...} for columns that
# need more precision, and percent_cols={"col", ...} for columns to show as a
# percentage (e.g. 0.393 -> "39.3%") rather than a raw decimal.
def style_df(df, decimals=1, decimal_overrides=None, percent_cols=None):
    decimal_overrides = decimal_overrides or {}
    percent_cols = set(percent_cols or [])

    def _make_fmt(nd, as_pct):
        def _fmt_num(x):
            if pd.isna(x):
                return ""
            val = x * 100 if as_pct else x
            suffix = "%" if as_pct else ""
            return f"({abs(val):,.{nd}f}{suffix})" if val < 0 else f"{val:,.{nd}f}{suffix}"
        return _fmt_num

    numeric_cols = df.select_dtypes(include="number").columns
    formatters = {
        c: _make_fmt(decimal_overrides.get(c, decimals), c in percent_cols)
        for c in numeric_cols
    }
    return (
        df.style
        .format(formatters)
        .set_properties(**{"text-align": "center", "white-space": "nowrap"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "center"), ("white-space", "nowrap")]}])
    )

In [4]:
CACHE_DIR = Path("S:/Structured Credit/Matt Stuff/quantifi_dsp_repos/delta_hedged_option_pnl/cache")
CACHE_DIR.mkdir(exist_ok=True)

TABLE = "teams.structured_credit_abhutra.credit_index_prices_gold"

In [5]:
credit_index_df = get_delta_table_mm(
    TABLE,
    CACHE_DIR,
    where="num_off_the_run IN (0, 1) AND index_tenor = '5Y' AND index_ticker IN ('IG','HY','Main','XO')",
    filename="credit_index_prices_gold_on_the_run_5y.parquet",
)
credit_index_df.head(30)

[cache] credit_index_prices_gold_on_the_run_5y.parquet is current — loading from disk


,published_date,index_ticker,index_name,index_tenor,index_series,index_version,composite_spread,composite_price,traded_version,valid_price,...,t_index_loss,t_index_recovery,traded_version_mtm_pnl,traded_version_accrued_pnl,traded_version_total_pnl,otr_version,num_off_the_run,otr_version_pnl,traded_version_spread_chg,otr_spread_change
0,2026-09-21,HY,HY46,5Y,46,3,302.298823,107.717091,Y,Y,...,NaN,NaN,0.002051,0.000408,0.002459,Y,0,0.002459,-5.683972,-5.683972
1,2026-09-21,HY,HY45,5Y,45,4,285.245355,107.697191,Y,Y,...,NaN,NaN,0.001881,0.000404,0.002286,N,1,NaN,-5.800122,NaN
2,2026-09-21,IG,IG47,5Y,47,1,56.133500,102.016443,Y,Y,...,NaN,NaN,NaN,NaN,NaN,Y,0,0.000469,NaN,-1.075492
3,2026-09-21,IG,IG46,5Y,46,1,49.569208,102.129849,Y,Y,...,NaN,NaN,0.000386,0.000083,0.000469,N,1,NaN,-1.075492,NaN
4,2026-09-21,Main,Main46,5Y,46,1,57.808076,102.008965,Y,Y,...,NaN,NaN,NaN,NaN,NaN,Y,0,0.000279,NaN,-0.605273
5,2026-09-21,Main,Main45,5Y,45,1,51.915876,102.096928,Y,Y,...,NaN,NaN,0.000196,0.000083,0.000279,N,1,NaN,-0.605273,NaN
6,2026-09-21,XO,XO46,5Y,46,1,284.200464,109.348071,Y,Y,...,NaN,NaN,NaN,NaN,NaN,Y,0,0.005892,NaN,-13.136411
7,2026-09-21,XO,XO45,5Y,45,1,246.240464,110.276618,Y,Y,...,NaN,NaN,0.005475,0.000417,0.005892,N,1,NaN,-13.136411,NaN
8,2026-09-18,HY,HY46,5Y,46,3,307.982795,107.507830,Y,Y,...,NaN,NaN,-0.000685,0.000136,-0.000549,Y,0,-0.000549,1.843334,1.843334
9,2026-09-18,HY,HY45,5Y,45,4,291.045477,107.503236,Y,Y,...,NaN,NaN,-0.000665,0.000135,-0.000530,N,1,NaN,1.922034,NaN


In [6]:
# --- Pull the raw CDX option table ---
OPTION_TABLE = "teams.structured_credit_abhutra.cdx_option_transform_raw_silver"

option_raw_df = get_delta_table_mm(
    OPTION_TABLE,
    CACHE_DIR,
    filename="cdx_option_transform_raw_silver.parquet",
)

[cache] cdx_option_transform_raw_silver.parquet is current — loading from disk


In [7]:
# Quantifi Toolkit imports for backtest pricing (CDX index options)
from Quantifi.Toolkit.Base import (
    Dt,
    MarketQuote,
    PayerReceiver,
    OptionStyle,
    CDXOptionModelType,
    CDXOptionModelParam,
    CDXOptionModelData,
)
from Quantifi.Toolkit.Products import CDXOption
from Quantifi.Toolkit.Products.StandardProductTerms import CreditIndexTerms
from Quantifi.Toolkit.Curves import DiscountCurve
from Quantifi.Toolkit.Pricers import CreditIndexOptionPricer, CreditIndexOptionPricerFactory
from Quantifi.Toolkit.Calibrators.Volatilities import CalibratedVolatilitySurface

# IG46 = CDX.NA.IG Series 46, standard 5Y underlying index terms
ig46_cdx = CreditIndexTerms.GetCDX("CDX.NA.IG", 46, 5)
ig46_cdx

<CLR: CDX.NA.IG.46 5 Years>

In [8]:
option_trade_dt = '2026-06-01'
option_index = 'IG46'
option_type = 'payer'
option_expiry = '2026-09-16'
option_strike = 55
option_notional = 250_000_000
option_direction = 'sell'
option_hedging_strat = 'daily'

DELTA_REBALANCE_THRESHOLD = 10_000_000   # IG46-equivalent index notional
DELTA_TRADE_LOT = 1_000_000              # round delta-hedge trades to the nearest lot

In [ ]:
# --- Join option prices/greeks (Markit) + index spreads for this trade's life ---
# Columns pulled straight from the vendor tables (no re-derivation) are suffixed
# _mkit. Anything re-derived via the Quantifi pricer gets a _qfi suffix in the
# next cell. Renamed to their final short names right here, at creation.
import re
import datetime as dt

# option_index like 'IG46' -> ticker 'IG', series 46
_m = re.match(r"([A-Za-z]+)(\d+)", option_index)
opt_index_ticker, opt_index_series = _m.group(1), int(_m.group(2))

trade_start = dt.date.fromisoformat(option_trade_dt)
trade_expiry = dt.date.fromisoformat(option_expiry)

# --- option leg (Markit CDX option table): this exact strike/expiry, one row per published_date ---
option_mask = (
    (option_raw_df["index_ticker"] == opt_index_ticker)
    & (option_raw_df["index_series"] == opt_index_series)
    & (option_raw_df["option_expiry"] == trade_expiry)
    & (option_raw_df["strike_level"] == option_strike)
    & (option_raw_df["published_date"] >= trade_start)
    & (option_raw_df["published_date"] <= trade_expiry)
)
price_col = "payer_mid" if option_type == "payer" else "receiver_mid"
delta_col = "delta_payer" if option_type == "payer" else "delta_receiver"

option_leg = (
    option_raw_df.loc[option_mask, ["published_date", price_col, delta_col, "vega_spread", "iv_spread_mid", "reference_spread"]]
    .rename(columns={
        "published_date": "date",
        price_col: "opx_prc",
        delta_col: "delta_mkit",
        "vega_spread": "vega_mkit",
        "iv_spread_mid": "iv_mkit",
        "reference_spread": "idx_spd_vol",
    })
    .sort_values("date")
    .reset_index(drop=True)
)

# --- index leg: same ticker/series/5Y tenor, same date window, from the gold table.
# idx_pnl_mkit (traded_version_total_pnl) - NOTIONAL BASIS UNCONFIRMED: this is the
# gold table's own traded-version PnL, not yet scaled to our delta_position. Treat
# as reference only until we confirm what notional it's computed on. ---
index_mask = (
    (credit_index_df["index_ticker"] == opt_index_ticker)
    & (credit_index_df["index_series"] == opt_index_series)
    & (credit_index_df["index_tenor"] == "5Y")
    & (credit_index_df["published_date"] >= trade_start)
    & (credit_index_df["published_date"] <= trade_expiry)
)

index_leg = (
    credit_index_df.loc[index_mask, ["published_date", "composite_spread", "traded_version_total_pnl"]]
    .rename(columns={
        "published_date": "date",
        "composite_spread": "idx_spd_mkit",
        "traded_version_total_pnl": "idx_pnl_mkit",
    })
)

pnl_df = option_leg.merge(index_leg, on="date", how="left").sort_values("date").reset_index(drop=True)
pnl_df = pnl_df[[
    "date", "opx_prc", "iv_mkit", "idx_spd_vol",
    "delta_mkit", "vega_mkit",
    "idx_pnl_mkit", "idx_spd_mkit",
]]

# signed option notional: negative when we're the seller
position_sign = -1 if option_direction == "sell" else 1
pnl_df.insert(1, "opx_ntl", position_sign * option_notional)

In [10]:
# --- _qfi columns: re-derive vol + all greeks each day from just opx_prc and
# idx_spd_vol, via the Quantifi pricer. Nothing here reuses a Markit greek -
# only the price and spread inputs are shared. Named directly with final short
# names (no separate rename step).
#
# ASSUMPTION: flat 4% discount curve (per-date AsOf) instead of a calibrated SOFR
# curve, per your simplification request. Model/settings match
# single_option_pricing_example.ipynb.
FLAT_DISCOUNT_RATE = 0.04

option_side = PayerReceiver.Payer if option_type == "payer" else PayerReceiver.Receiver

# Standard CDX Option model settings (matches Quantifi's own CDX Options sample):
# full-spread model, spot spread adjusted to a forward via the market method.
model_data = CDXOptionModelData()
model_data.Choice = (
    CDXOptionModelParam.AdjustSpread
    | CDXOptionModelParam.AdjustSpreadByMarketMethod
    | CDXOptionModelParam.ForceFlatMarketCurve
    | CDXOptionModelParam.ForceCleanValue
    | CDXOptionModelParam.MarketPayoff
    | CDXOptionModelParam.HandleIndexFactors
)


cdx_option_product = CDXOption(
    effective=ig46_cdx.Effective,
    ccy=ig46_cdx.Ccy,
    underlying=ig46_cdx,
    expiration=Dt(int(trade_expiry.strftime("%Y%m%d"))),
    type=option_side,
    style=OptionStyle.European,
    strike=option_strike / 10000.0,
    strikeIsPrice=False,
    indexFactor=1.0,
)


def _build_pricer(px_date, spot_quote_i, discount_curve_i, vol_surface_i):
    return CreditIndexOptionPricer(
        cdx_option_product,
        px_date,
        Dt.AddDays(px_date, 1, None),   # settle date
        discount_curve_i,
        spot_quote_i,
        None,                      # cdxSettleDate
        ig46_cdx.Recovery,
        0,                         # basketSize
        None,                      # survivalCurves
        1.0,                       # currentFactor
        float("nan"),              # existingLossToInclude
        CDXOptionModelType.FullSpread,
        model_data,
        vol_surface_i,
    )


iv_qfi, delta_qfi, gamma_qfi, vega_qfi, theta_qfi = [], [], [], [], []
for _, row in pnl_df.iterrows():
    px_date = Dt(int(row["date"].strftime("%Y%m%d")))
    discount_curve_i = DiscountCurve(px_date, FLAT_DISCOUNT_RATE)
    spot_quote_i = MarketQuote(row["idx_spd_vol"] / 10000.0, ig46_cdx.QuotingConvention)

    # seed pricer (vol value here is a throwaway - only used to bootstrap ImplyVolatility)
    seed_vol_surface = CalibratedVolatilitySurface.FromFlatVolatility(px_date, 0.30)
    seed_pricer = _build_pricer(px_date, spot_quote_i, discount_curve_i, seed_vol_surface)

    market_fair_value = row["opx_prc"] / 10000.0   # decimal price, same units as FairPrice()
    iv_i = seed_pricer.ImplyVolatility(market_fair_value)

    vol_surface_i = CalibratedVolatilitySurface.FromFlatVolatility(px_date, iv_i)
    pricer_i = _build_pricer(px_date, spot_quote_i, discount_curve_i, vol_surface_i)

    iv_qfi.append(iv_i)
    delta_qfi.append(CreditIndexOptionPricerFactory.MarketDelta(pricer_i))
    gamma_qfi.append(CreditIndexOptionPricerFactory.MarketGamma(pricer_i))
    vega_qfi.append(CreditIndexOptionPricerFactory.Vega(pricer_i))
    theta_qfi.append(CreditIndexOptionPricerFactory.MarketTheta(pricer_i))

pnl_df["iv_qfi"] = iv_qfi
pnl_df["delta_qfi"] = delta_qfi
pnl_df["gamma_qfi"] = gamma_qfi
pnl_df["vega_qfi"] = vega_qfi
pnl_df["theta_qfi"] = theta_qfi

# index-equivalent RISK notional: -1 x (our signed option notional x model delta).
# opx_ntl x delta_qfi gives the hedge notional (the offsetting index position that
# neutralizes the option's delta); flipping sign here instead expresses our own
# directional risk position - i.e. the index-equivalent exposure our option
# position itself carries, same sign convention as being long/short the index.
pnl_df.insert(2, "idx_eqv_ntl", -1 * pnl_df["opx_ntl"] * pnl_df["delta_qfi"])
pnl_df

,date,opx_ntl,idx_eqv_ntl,opx_prc,idx_spd_vol,delta_mkit,vega_mkit,iv_mkit,idx_pnl_mkit,idx_spd_mkit,iv_qfi,delta_qfi,gamma_qfi,vega_qfi,theta_qfi
0,2026-06-01,-250000000,1.340492e+08,18.635763,50.773292,0.520801,0.506929,0.394563,-0.000070,50.773292,0.395006,0.536197,0.286110,5.041545e-05,-0.000017
1,2026-06-02,-250000000,1.312172e+08,18.075178,50.476583,0.510463,0.501690,0.399546,0.000135,50.476583,0.401072,0.524869,0.288767,4.991857e-05,-0.000016
2,2026-06-03,-250000000,1.380285e+08,20.025733,51.264169,0.537595,0.504556,0.404956,-0.000344,51.264169,0.405660,0.552114,0.275066,5.021369e-05,-0.000017
3,2026-06-04,-250000000,1.341485e+08,18.924647,50.859455,0.522754,0.499314,0.404847,0.000187,50.859455,0.406711,0.536594,0.282415,4.970572e-05,-0.000017
4,2026-06-05,-250000000,1.484131e+08,23.444535,52.540654,0.579808,0.503532,0.414044,-0.000747,52.540654,0.415805,0.593653,0.252311,5.010690e-05,-0.000018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,2026-09-10,-250000000,5.413948e+07,1.057366,51.945500,0.168239,0.072857,0.416808,-0.000556,51.945500,0.418248,0.216558,0.777079,7.377237e-06,-0.000027
74,2026-09-11,-250000000,3.662669e+07,0.554225,51.443201,0.106721,0.048014,0.423604,0.000172,51.443201,0.425809,0.146507,0.847263,4.893555e-06,-0.000021
75,2026-09-14,-250000000,2.727662e+07,0.235944,51.951000,0.068082,0.021770,0.483109,-0.000203,51.951000,0.493867,0.109106,0.891288,2.204107e-06,-0.000021
76,2026-09-15,-250000000,1.211099e+07,0.037246,52.197667,0.019561,0.005483,0.457617,-0.000101,52.197667,0.469996,0.048444,0.952182,5.636235e-07,-0.000004


In [11]:
# --- shared style_df() formatting spec, reused by every pnl_df display below ---
PNL_DECIMALS = {
    "gamma_qfi": 3, "vega_qfi": 3, "theta_qfi": 3,
    "vega_mkit": 3, "idx_spd_vol": 3, "idx_spd_mkit": 3,
    "idx_eqv_ntl": 0, "idx_pnl_mkit": 4,
    "traded_opx": 0, "traded_dlt": 0, "opx_position": 0, "delta_position": 0,
    "idx_pnl": 0, "opx_pnl": 0, "total_pnl": 0, "cum_pnl": 0,
}
PNL_PERCENT_COLS = {"delta_mkit", "iv_mkit", "iv_qfi", "delta_qfi"}

In [12]:
# --- threshold-rebalanced delta hedge: build the day-by-day trade/position ledger ---
# Rules:
#   - Day 1 (trade inception): execute the option (opx_ntl, at that day's opx_prc)
#     AND fully delta-hedge with the index (at that day's idx_spd_vol) - the initial
#     hedge trade happens unconditionally, not gated by the threshold.
#   - The option is never re-traded after day 1 (opx_position stays flat at opx_ntl).
#   - Every day after day 1, we only trade the index when the net package's delta
#     equivalent - idx_eqv_ntl (the option's current risk) + delta_position (index
#     held from prior trades) - exceeds DELTA_REBALANCE_THRESHOLD in absolute value.
#     When it does, we trade enough index to bring the net back to exactly zero,
#     rounded to the nearest DELTA_TRADE_LOT.
#   - All trades are assumed done at that day's closing levels.
traded_opx_list, traded_dlt_list = [], []
opx_position_list, delta_position_list = [], []

opx_pos = 0.0
delta_pos = 0.0

for pos, (_, row) in enumerate(pnl_df.iterrows()):
    is_trade_day = (pos == 0)

    traded_opx = row["opx_ntl"] if is_trade_day else 0.0
    opx_pos += traded_opx

    net_pkg_delta = row["idx_eqv_ntl"] + delta_pos
    rebalance = is_trade_day or abs(net_pkg_delta) > DELTA_REBALANCE_THRESHOLD
    traded_dlt = round(-net_pkg_delta / DELTA_TRADE_LOT) * DELTA_TRADE_LOT if rebalance else 0.0
    delta_pos += traded_dlt

    traded_opx_list.append(traded_opx)
    traded_dlt_list.append(traded_dlt)
    opx_position_list.append(opx_pos)
    delta_position_list.append(delta_pos)

pnl_df["traded_opx"] = traded_opx_list
pnl_df["traded_dlt"] = traded_dlt_list
pnl_df["opx_position"] = opx_position_list
pnl_df["delta_position"] = delta_position_list

style_df(pnl_df, decimal_overrides=PNL_DECIMALS, percent_cols=PNL_PERCENT_COLS)

,date,opx_ntl,idx_eqv_ntl,opx_prc,idx_spd_vol,delta_mkit,vega_mkit,iv_mkit,idx_pnl_mkit,idx_spd_mkit,iv_qfi,delta_qfi,gamma_qfi,vega_qfi,theta_qfi,traded_opx,traded_dlt,opx_position,delta_position
0,2026-06-01,"(250,000,000.0)","134,049,171",18.6,50.773,52.1%,0.507,39.5%,(0.0001),50.773,39.5%,53.6%,0.286,0.000,(0.000),"(250,000,000)","(134,000,000)","(250,000,000)","(134,000,000)"
1,2026-06-02,"(250,000,000.0)","131,217,247",18.1,50.477,51.0%,0.502,40.0%,0.0001,50.477,40.1%,52.5%,0.289,0.000,(0.000),0,0,"(250,000,000)","(134,000,000)"
2,2026-06-03,"(250,000,000.0)","138,028,464",20.0,51.264,53.8%,0.505,40.5%,(0.0003),51.264,40.6%,55.2%,0.275,0.000,(0.000),0,0,"(250,000,000)","(134,000,000)"
3,2026-06-04,"(250,000,000.0)","134,148,459",18.9,50.859,52.3%,0.499,40.5%,0.0002,50.859,40.7%,53.7%,0.282,0.000,(0.000),0,0,"(250,000,000)","(134,000,000)"
4,2026-06-05,"(250,000,000.0)","148,413,144",23.4,52.541,58.0%,0.504,41.4%,(0.0007),52.541,41.6%,59.4%,0.252,0.000,(0.000),0,"(14,000,000)","(250,000,000)","(148,000,000)"
5,2026-06-08,"(250,000,000.0)","142,600,493",21.4,51.950,55.5%,0.493,41.7%,0.0003,51.950,41.6%,57.0%,0.266,0.000,(0.000),0,0,"(250,000,000)","(148,000,000)"
6,2026-06-09,"(250,000,000.0)","138,947,223",20.4,51.557,54.3%,0.489,41.8%,0.0002,51.557,41.9%,55.6%,0.272,0.000,(0.000),0,0,"(250,000,000)","(148,000,000)"
7,2026-06-10,"(250,000,000.0)","147,517,261",22.9,52.592,57.7%,0.490,41.8%,(0.0004),52.592,42.0%,59.0%,0.256,0.000,(0.000),0,0,"(250,000,000)","(148,000,000)"
8,2026-06-11,"(250,000,000.0)","132,512,845",18.4,50.903,51.7%,0.481,41.7%,0.0008,50.903,41.9%,53.0%,0.285,0.000,(0.000),0,"15,000,000","(250,000,000)","(133,000,000)"
9,2026-06-12,"(250,000,000.0)","128,364,844",17.2,50.491,49.9%,0.475,41.6%,0.0003,50.491,41.8%,51.3%,0.294,0.000,(0.000),0,0,"(250,000,000)","(133,000,000)"


In [13]:
# --- daily PnL, both legs, plus total and cumulative ---
# idx_pnl: idx_pnl_mkit (a per-dollar daily return) x the index position we were
# actually carrying INTO that day - i.e. delta_position as of the prior day's close,
# since we only rebalance at EOD so that's the position held all through today.
# opx_pnl: day-over-day change in opx_prc (quoted in cents/bp of notional) x the
# option position carried into today, /10000 to convert cents to a fraction of
# notional. opx_position never changes after day 1, so this is just the option's
# mark-to-market move on a fixed notional - except day 1 itself, where there was no
# prior position (shift(1) -> 0), so day-1 PnL on both legs is correctly 0.
pnl_df["idx_pnl"] = pnl_df["idx_pnl_mkit"] * pnl_df["delta_position"].shift(1).fillna(0)
pnl_df["opx_pnl"] = pnl_df["opx_prc"].diff().fillna(0) / 10000 * pnl_df["opx_position"].shift(1).fillna(0)
pnl_df["total_pnl"] = pnl_df["idx_pnl"] + pnl_df["opx_pnl"]
pnl_df["cum_pnl"] = pnl_df["total_pnl"].cumsum()

# move the trade/position/pnl ledger columns (the last 8 added) to sit right after idx_spd_vol
_cols = pnl_df.columns.tolist()
_ledger_cols, _rest = _cols[-8:], _cols[:-8]
_insert_at = _rest.index("idx_spd_vol") + 1
pnl_df = pnl_df[_rest[:_insert_at] + _ledger_cols + _rest[_insert_at:]]

style_df(pnl_df, decimal_overrides=PNL_DECIMALS, percent_cols=PNL_PERCENT_COLS)

,date,opx_ntl,idx_eqv_ntl,opx_prc,idx_spd_vol,traded_opx,traded_dlt,opx_position,delta_position,idx_pnl,opx_pnl,total_pnl,cum_pnl,delta_mkit,vega_mkit,iv_mkit,idx_pnl_mkit,idx_spd_mkit,iv_qfi,delta_qfi,gamma_qfi,vega_qfi,theta_qfi
0,2026-06-01,"(250,000,000.0)","134,049,171",18.6,50.773,"(250,000,000)","(134,000,000)","(250,000,000)","(134,000,000)",-0,0,0,0,52.1%,0.507,39.5%,(0.0001),50.773,39.5%,53.6%,0.286,0.000,(0.000)
1,2026-06-02,"(250,000,000.0)","131,217,247",18.1,50.477,0,0,"(250,000,000)","(134,000,000)","(18,058)","14,015","(4,043)","(4,043)",51.0%,0.502,40.0%,0.0001,50.477,40.1%,52.5%,0.289,0.000,(0.000)
2,2026-06-03,"(250,000,000.0)","138,028,464",20.0,51.264,0,0,"(250,000,000)","(134,000,000)","46,151","(48,764)","(2,613)","(6,656)",53.8%,0.505,40.5%,(0.0003),51.264,40.6%,55.2%,0.275,0.000,(0.000)
3,2026-06-04,"(250,000,000.0)","134,148,459",18.9,50.859,0,0,"(250,000,000)","(134,000,000)","(25,060)","27,527","2,467","(4,189)",52.3%,0.499,40.5%,0.0002,50.859,40.7%,53.7%,0.282,0.000,(0.000)
4,2026-06-05,"(250,000,000.0)","148,413,144",23.4,52.541,0,"(14,000,000)","(250,000,000)","(148,000,000)","100,071","(112,997)","(12,927)","(17,115)",58.0%,0.504,41.4%,(0.0007),52.541,41.6%,59.4%,0.252,0.000,(0.000)
5,2026-06-08,"(250,000,000.0)","142,600,493",21.4,51.950,0,0,"(250,000,000)","(148,000,000)","(39,536)","52,199","12,663","(4,453)",55.5%,0.493,41.7%,0.0003,51.950,41.6%,57.0%,0.266,0.000,(0.000)
6,2026-06-09,"(250,000,000.0)","138,947,223",20.4,51.557,0,0,"(250,000,000)","(148,000,000)","(29,097)","24,574","(4,524)","(8,977)",54.3%,0.489,41.8%,0.0002,51.557,41.9%,55.6%,0.272,0.000,(0.000)
7,2026-06-10,"(250,000,000.0)","147,517,261",22.9,52.592,0,0,"(250,000,000)","(148,000,000)","64,504","(62,607)","1,897","(7,080)",57.7%,0.490,41.8%,(0.0004),52.592,42.0%,59.0%,0.256,0.000,(0.000)
8,2026-06-11,"(250,000,000.0)","132,512,845",18.4,50.903,0,"15,000,000","(250,000,000)","(133,000,000)","(116,852)","111,147","(5,705)","(12,785)",51.7%,0.481,41.7%,0.0008,50.903,41.9%,53.0%,0.285,0.000,(0.000)
9,2026-06-12,"(250,000,000.0)","128,364,844",17.2,50.491,0,0,"(250,000,000)","(133,000,000)","(33,435)","30,087","(3,348)","(16,134)",49.9%,0.475,41.6%,0.0003,50.491,41.8%,51.3%,0.294,0.000,(0.000)


In [ ]:
# --- realized vol of the index spread over the trade's life ---
# stdev(ln(spread_t / spread_t-1)) annualized by sqrt(252). Uses idx_spd_vol (the
# option table's own reference spread) since that's the same series driving opx_prc
# and iv_qfi/delta_qfi above - keeps this apples-to-apples with the rest of the sheet.
import numpy as np

log_returns = np.log(pnl_df["idx_spd_vol"] / pnl_df["idx_spd_vol"].shift(1))
realized_vol = log_returns.std() * np.sqrt(252)

print(f"Realized vol (annualized):     {realized_vol:.2%}")
print(f"Avg iv_mkit over the period:   {pnl_df['iv_mkit'].mean():.2%}")
print(f"Avg iv_qfi over the period:    {pnl_df['iv_qfi'].mean():.2%}")
print()
# since we never re-trade the option after day 1, only the vol we actually traded
# AT inception matters for this position - the period average above is just context.
print(f"Starting (day-1) iv_mkit:      {pnl_df['iv_mkit'].iloc[0]:.2%}")
print(f"Starting (day-1) iv_qfi:       {pnl_df['iv_qfi'].iloc[0]:.2%}")

In [ ]:
# --- generalize: run the exact same exercise across every in-scope expiry for
# IG46 down through IG41 (whatever data we have). ATM strike picked off that day's
# fwd_spread; trade date = 1st of the month 3 months before expiry, snapped to the
# nearest actual published date. Constrained to num_off_the_run <= 1 throughout -
# both at trade inception (skip if the series wasn't yet <=1-off by the target
# date) and through to expiry (skip if it rolled to 2-off mid-trade). ---
MAX_SNAP_DAYS = 5   # if the 1st-of-month target isn't in the data within this many
                     # days, the series wasn't <=1-off-the-run yet - skip it, don't
                     # silently shorten the holding period.
SERIES_LIST = [46, 45, 44, 43, 42, 41]


def month_before(d, months):
    m = d.month - months
    y = d.year
    while m <= 0:
        m += 12
        y -= 1
    return dt.date(y, m, 1)


ig_otr_or_1off = option_raw_df[(option_raw_df["index_ticker"] == "IG") & (option_raw_df["num_off_the_run"] <= 1)]

backtest_params = []
skipped = []
for series in SERIES_LIST:
    chain = ig_otr_or_1off[ig_otr_or_1off["index_series"] == series]
    if chain.empty:
        skipped.append((series, None, "no <=1-off-the-run data at all"))
        continue
    last_available = chain["published_date"].max()
    candidate_expiries = sorted(e for e in chain["option_expiry"].unique() if e <= last_available)

    for exp in candidate_expiries:
        target = month_before(exp, 3)
        exp_chain = chain[chain["option_expiry"] == exp]
        on_or_after = exp_chain[exp_chain["published_date"] >= target]["published_date"]
        if on_or_after.empty:
            skipped.append((series, exp, f"no data on/after target {target}"))
            continue
        trade_dt = on_or_after.min()
        if (trade_dt - target).days > MAX_SNAP_DAYS:
            skipped.append((series, exp, f"target {target}, nearest available {trade_dt} - not yet <=1-off"))
            continue
        if exp_chain["published_date"].max() < exp:
            skipped.append((series, exp, f"chain ends {exp_chain['published_date'].max()} before expiry - rolled to 2-off mid-trade"))
            continue

        day_chain = exp_chain[exp_chain["published_date"] == trade_dt]
        fwd = day_chain["fwd_spread"].iloc[0]
        atm_strike = day_chain.loc[(day_chain["strike_level"] - fwd).abs().idxmin(), "strike_level"]
        backtest_params.append({"series": series, "expiry": exp, "trade_dt": trade_dt, "strike": atm_strike, "fwd_spread": fwd})

print(f"Running {len(backtest_params)} trades across series {SERIES_LIST}; skipped {len(skipped)}:")
for series, exp, reason in skipped:
    print(f"  skip IG{series} {exp} - {reason}")

In [ ]:
# --- run_ig_backtest: same exercise as the single-trade cells above, but as a
# function of (trade_dt, expiry, strike, index_series). Builds the correct
# CDX.NA.IG series object per call (cached), reuses model_data/FLAT_DISCOUNT_RATE
# from earlier cells. Both option and index legs are constrained to
# num_off_the_run <= 1. Skips gamma/vega/theta (not needed for this summary) to
# keep the per-day Quantifi call count down now that this runs many times.
_cdx_cache = {}


def _get_cdx(series):
    if series not in _cdx_cache:
        _cdx_cache[series] = CreditIndexTerms.GetCDX("CDX.NA.IG", series, 5)
    return _cdx_cache[series]


def run_ig_backtest(trade_dt, expiry, strike, index_series, notional=250_000_000, direction="sell", option_type="payer"):
    cdx = _get_cdx(index_series)

    option_mask = (
        (option_raw_df["index_ticker"] == "IG")
        & (option_raw_df["index_series"] == index_series)
        & (option_raw_df["num_off_the_run"] <= 1)
        & (option_raw_df["option_expiry"] == expiry)
        & (option_raw_df["strike_level"] == strike)
        & (option_raw_df["published_date"] >= trade_dt)
        & (option_raw_df["published_date"] <= expiry)
    )
    price_col = "payer_mid" if option_type == "payer" else "receiver_mid"

    df = (
        option_raw_df.loc[option_mask, ["published_date", price_col, "reference_spread", "iv_spread_mid"]]
        .rename(columns={
            "published_date": "date",
            price_col: "opx_prc",
            "reference_spread": "idx_spd_vol",
            "iv_spread_mid": "iv_mkit",
        })
        .sort_values("date")
        .reset_index(drop=True)
    )

    index_mask = (
        (credit_index_df["index_ticker"] == "IG")
        & (credit_index_df["index_series"] == index_series)
        & (credit_index_df["index_tenor"] == "5Y")
        & (credit_index_df["num_off_the_run"] <= 1)
        & (credit_index_df["published_date"] >= trade_dt)
        & (credit_index_df["published_date"] <= expiry)
    )
    index_leg = (
        credit_index_df.loc[index_mask, ["published_date", "traded_version_total_pnl"]]
        .rename(columns={"published_date": "date", "traded_version_total_pnl": "idx_pnl_mkit"})
    )
    df = df.merge(index_leg, on="date", how="left").sort_values("date").reset_index(drop=True)

    position_sign = -1 if direction == "sell" else 1
    opx_ntl = position_sign * notional

    option_side = PayerReceiver.Payer if option_type == "payer" else PayerReceiver.Receiver
    cdx_option_product = CDXOption(
        effective=cdx.Effective,
        ccy=cdx.Ccy,
        underlying=cdx,
        expiration=Dt(int(expiry.strftime("%Y%m%d"))),
        type=option_side,
        style=OptionStyle.European,
        strike=strike / 10000.0,
        strikeIsPrice=False,
        indexFactor=1.0,
    )

    def _build_pricer(px_date, spot_quote_i, discount_curve_i, vol_surface_i):
        return CreditIndexOptionPricer(
            cdx_option_product, px_date, Dt.AddDays(px_date, 1, None),
            discount_curve_i, spot_quote_i, None, cdx.Recovery, 0, None,
            1.0, float("nan"), CDXOptionModelType.FullSpread, model_data, vol_surface_i,
        )

    iv_qfi, delta_qfi = [], []
    for _, row in df.iterrows():
        px_date = Dt(int(row["date"].strftime("%Y%m%d")))
        discount_curve_i = DiscountCurve(px_date, FLAT_DISCOUNT_RATE)
        spot_quote_i = MarketQuote(row["idx_spd_vol"] / 10000.0, cdx.QuotingConvention)

        seed_pricer = _build_pricer(px_date, spot_quote_i, discount_curve_i,
                                     CalibratedVolatilitySurface.FromFlatVolatility(px_date, 0.30))
        iv_i = seed_pricer.ImplyVolatility(row["opx_prc"] / 10000.0)

        pricer_i = _build_pricer(px_date, spot_quote_i, discount_curve_i,
                                  CalibratedVolatilitySurface.FromFlatVolatility(px_date, iv_i))
        iv_qfi.append(iv_i)
        delta_qfi.append(CreditIndexOptionPricerFactory.MarketDelta(pricer_i))

    df["iv_qfi"] = iv_qfi
    df["delta_qfi"] = delta_qfi
    df["idx_eqv_ntl"] = -1 * opx_ntl * df["delta_qfi"]

    traded_dlt_list, opx_position_list, delta_position_list = [], [], []
    opx_pos, delta_pos = 0.0, 0.0
    for pos, (_, row) in enumerate(df.iterrows()):
        is_trade_day = (pos == 0)
        opx_pos += opx_ntl if is_trade_day else 0.0
        net_pkg_delta = row["idx_eqv_ntl"] + delta_pos
        rebalance = is_trade_day or abs(net_pkg_delta) > DELTA_REBALANCE_THRESHOLD
        traded_dlt = round(-net_pkg_delta / DELTA_TRADE_LOT) * DELTA_TRADE_LOT if rebalance else 0.0
        delta_pos += traded_dlt
        opx_position_list.append(opx_pos)
        delta_position_list.append(delta_pos)

    df["opx_position"] = opx_position_list
    df["delta_position"] = delta_position_list

    df["idx_pnl"] = df["idx_pnl_mkit"] * df["delta_position"].shift(1).fillna(0)
    df["opx_pnl"] = df["opx_prc"].diff().fillna(0) / 10000 * df["opx_position"].shift(1).fillna(0)
    df["total_pnl"] = df["idx_pnl"] + df["opx_pnl"]
    df["cum_pnl"] = df["total_pnl"].cumsum()

    log_returns = np.log(df["idx_spd_vol"] / df["idx_spd_vol"].shift(1))
    realized_vol = log_returns.std() * np.sqrt(252)

    return {
        "series": index_series,
        "expiry": expiry,
        "trade_dt": trade_dt,
        "strike": strike,
        "starting_iv_mkit": df["iv_mkit"].iloc[0],
        "starting_iv_qfi": df["iv_qfi"].iloc[0],
        "realized_vol": realized_vol,
        "cum_pnl": df["cum_pnl"].iloc[-1],
        "detail": df,
    }

In [ ]:
# --- run it across all in-scope IG41-46 expiries and summarize ---
ig_results = [
    run_ig_backtest(p["trade_dt"], p["expiry"], p["strike"], p["series"])
    for p in backtest_params
]

ig_summary = pd.DataFrame(ig_results)[
    ["series", "expiry", "trade_dt", "strike", "starting_iv_mkit", "starting_iv_qfi", "realized_vol", "cum_pnl"]
]

style_df(
    ig_summary,
    decimal_overrides={"strike": 1, "cum_pnl": 0, "series": 0},
    percent_cols={"starting_iv_mkit", "starting_iv_qfi", "realized_vol"},
)

In [ ]:
# --- cum_pnl distribution across all in-scope trades ---
cum_pnl = ig_summary["cum_pnl"]
print(f"min:    {cum_pnl.min():,.0f}")
print(f"25th:   {cum_pnl.quantile(0.25):,.0f}")
print(f"median: {cum_pnl.median():,.0f}")
print(f"mean:   {cum_pnl.mean():,.0f}")
print(f"75th:   {cum_pnl.quantile(0.75):,.0f}")
print(f"max:    {cum_pnl.max():,.0f}")